## Deep Research

業種を超えて使える、Agenticのクラシックなユースケースの1つです！これはとても大きな意味を持っています。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">商業的な意義</h2>
            <span style="color:#00bfff;">Deep Researchエージェントは、どんなビジネス領域にも、そして自分自身の日々の活動にも広く応用できます。ぜひ自分でも活用してみてください！
            </span>
        </td>
    </tr>
</table>

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from openai import AsyncOpenAI
import asyncio
import os
from IPython.display import display, Markdown
from messenger import send_email, push

In [ ]:
load_dotenv(override=True)

In [ ]:
# 定数

# メインモデルはGeminiに切り替え。ただしWebSearchTool()はOpenAIホストの専用ツールで
# GeminiのOpenAI互換エンドポイントでは使えないため、検索エージェントだけは
# 引き続きOpenAIのモデル(SEARCH_MODEL_NAME)を使う
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = AsyncOpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url=GEMINI_BASE_URL)
MODEL_NAME = OpenAIChatCompletionsModel(model="gemini-flash-latest", openai_client=gemini_client)
SEARCH_MODEL_NAME = "gpt-5.4-mini"  # WebSearchTool用。OpenAIのモデルのまま
set_tracing_disabled(True)  # GeminiのキーではOpenAIのトレースは使えないため

USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Deep Research Agentの戦略

ここでは、確実で堅牢な方法で進めていきます。

コードによってオーケストレーションを行います。プロセスの各ステップごとに、個別の`Runner.run()`呼び出しを行います。

各ステップでStructured Outputsを使用します。

## 4つのAgentを構築します:

1. Search Agent: Web上で情報を検索する
2. Planner Agent: 質問が与えられたら、行うべき検索のリストを考え出す
3. Writer Agent: しっかりとしたレポートを書く
4. Emailer Agent: メールを作成して送信する

そして、4つのAgentそれぞれについてRunner.run()を呼び出す、4つのPython関数を作ります。


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

OpenAIのクラウド上で管理された機能を実行するための、有料でお手軽な方法です。

彼らのドキュメントではこれらのツールが取り上げられていますが、コストがかかること、そしてOpenAIのエコシステムに縛られることになる点は覚えておく価値があります。

OpenAIは以下のhosted toolsを提供しています。

`WebSearchTool`は、エージェントがWebを検索できるようにします。  
`FileSearchTool`は、自分のOpenAI Vector Storesから情報を取得できるようにします。  
`CodeInterpreterTool`は、LLMがサンドボックス環境内でコードを実行できるようにします。  
`HostedMCPTool`は、リモートのMCPサーバーのツールをモデルに公開します。  
`ImageGenerationTool`は、プロンプトから画像を生成します。  
`ToolSearchTool`は、モデルが遅延読み込みのツール、namespace、あるいはhosted MCPサーバーを必要に応じて読み込めるようにします。  

### 重要な注意事項 - WebSearchToolのAPI料金について

現在、OpenAIのWebSearchToolは1回の呼び出しにつき1セントのコストがかかります。これは次の2つのラボで合計約1ドルほどになる可能性があります。他のプラットフォームでは無料または低コストの検索ツールを使用するので、コストが気になる場合はこれを実行せずスキップしてもかまいません。また、受講生のChristian W.が指摘してくれたのですが、OpenAIは1回の呼び出しに対して複数回分の検索料金を課すことがあるため、1回の呼び出しにつき1セントより高くなる場合もあります。

料金についてはこちらのToolsセクションをご覧ください: https://developers.openai.com/api/docs/pricing


In [ ]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""
task = "Most popular AI Agent frameworks in 2026"

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [ ]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=SEARCH_MODEL_NAME, model_settings=settings)

In [ ]:
result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

### いつものように、traceを見てみましょう

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### ここではStructured Outputsを使い、各フィールドの説明も加えます

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [ ]:
WebSearchPlan.model_json_schema()

In [ ]:
# WebSearchToolのコストについては上記の注意事項を参照してください

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:

result = await Runner.run(planner_agent, task)
result.final_output

## Agent 3: The Writer Agent

In [ ]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

## Agent 4: The email agent

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent successfully"

In [ ]:
send_email_tool.params_json_schema

In [ ]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## いよいよコードによるオーケストレーションです

次の2つの関数は、`Runner.run()`の呼び出しを通じてAgentを使い、検索を計画・実行します

In [ ]:
async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

次の2つの関数は、レポートを書いてメールで送信します

In [ ]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### いよいよ本番です！

In [ ]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

### いつものように、traceを見てみましょう

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">ここまでの進歩を祝して、そして1つお願いがあります</h2>
            <span style="color:#00cc00;">このコースにおいて、あなたは重要な節目に到達しました。最新のAgentフレームワークの1つを使って、価値あるAgentを作り上げたのです。スキルを高め、新しい商業的な可能性を切り開きました。この成功を、少し時間をとって祝ってください！<br/><br/>1つお願いしたいことがあります。もしこれに触れなければ、私の編集者に叱られてしまいます。もしUdemyでこのコースを評価していただけるなら、本当にありがたく思います。Udemyがこのコースをほかの人に見せるかどうかを判断する上で、これが最も重要な要素であり、大きな違いを生むのです。<br/><br/>そしてもう一つ、よろしければ<a href="https://www.linkedin.com/in/eddonner/">LinkedInで私とつながって</a>ください！コースでの進歩についてぜひ投稿してみてください。私をタグ付けしていただければ、コメントを入れてあなたの発信力を高めるお手伝いをします。
            </span>
        </td>
    </tr>